# Movie Recommendation System using LightFM

With LightFM, we would be able to approach the recommendation system with a hybrid approach of collaborative and cotent.

In [1]:
import pandas as pd 
# from sklearn.preprocessing import MultiLabelBinarizer
# from sklearn.metrics.pairwise import cosine_similarity
# from scipy.sparse import csr_matrix
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from scipy import sparse
# import os
# import boto3
# from dotenv import load_dotenv
import pickle


In [ ]:
load_dotenv()

bucket_name = os.getenv("AWS_BUCKET_NAME")
ratings_file = os.getenv("AWS_RATINGS_FILE")
models_file = os.getenv("AWS_MODEL_FILE")

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AWS_SECRET"),
    region_name=os.getenv("AWS_REGION")
)

s3.download_file(bucket_name, ratings_file, "ratings.csv")
s3.download_file(bucket_name, models_file, models_file)

In [2]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv('../BigMovieData/ml-32m/movies.csv')

In [3]:
personal_ratings = pd.read_csv("../personal_letterboxd/ratings.csv")
personal_ratings["Year"] = personal_ratings["Year"].astype(str)
personal_ratings['title'] = personal_ratings["Name"] + " (" + personal_ratings['Year'] + ")"
personal_ratings.head()

user_rating_merged = personal_ratings.merge(
    movies,
    left_on=["title"],
    right_on=["title"],
    how="inner"
)

user_rating_merged['userId'] = 200949
user_rating_merged.head()
final_user_rating = user_rating_merged[['userId', 'movieId', 'Rating']]
final_user_rating.rename(columns={"Rating": "rating"}, inplace=True)
ratings = pd.concat([ratings, final_user_rating], ignore_index= True)

# ratings = ratings.drop(columns="timestamp")
ratings['userId'] = ratings['userId'].astype(int)
ratings['movieId'] = ratings['movieId'].astype(int)
ratings['rating'] = ratings['rating'].astype(float)
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])
ratings = ratings.drop(columns="timestamp")

/tmp/ipykernel_14409/3654409920.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_user_rating.rename(columns={"Rating": "rating"}, inplace=True)


In [4]:
movie_id_to_genres = {}
all_genres = set()

for _, row in movies.iterrows():
    if isinstance(row['genres'], str) and row['genres'] != '(no genres listed)':
        genres = list(set(row['genres'].split('|')))  # remove duplicate genres
        movie_id_to_genres[row['movieId']] = genres
        all_genres.update(genres)
movie_id_to_genres = {int(k): v for k, v in movie_id_to_genres.items()}

In [5]:
# Movie IDs from ratings
ratings['movieId'] = ratings['movieId'] - 1
ratings_movie_ids = set(ratings['movieId'].unique())
for movie_id in ratings_movie_ids:
    if movie_id not in movie_id_to_genres:
        movie_id_to_genres[movie_id] = ["Unknown"]
# Movie IDs from the movies DataFrame (used in dataset.fit)
movies_movie_ids = set(movies['movieId'])        
adjusted_movie_id_to_genres = {movie_id - 1: genres for movie_id, genres in movie_id_to_genres.items()}


In [6]:
dataset = Dataset()
dataset.fit(
    users=ratings['userId'].unique(),
    items=ratings['movieId'].unique(),  # Ensure only rated movies
    item_features=set(g for genres in movie_id_to_genres.values() for g in genres)
)



In [7]:
ratings.head()

,userId,movieId,rating
0,1,16,4.0
1,1,24,1.0
2,1,28,2.0
3,1,29,5.0
4,1,31,5.0


In [8]:
_, item_map, _, _ = dataset.mapping()
# print("LightFM internal movie ID map:")
# print(item_map)
item_features_list = []
missing_features = []

# Create the internal mapping for movie_id_to_genres
for movie_id, genres in movie_id_to_genres.items():
    if movie_id in item_map:  # Ensure the movie_id is in the internal mapping
        internal_id = item_map[movie_id]
        item_features_list.append((str(internal_id), genres))
    else:
        missing_features.append(movie_id)

# Report missing movie IDs
if missing_features:
    print(f"Missing movie IDs: {missing_features}")
item_features_list = [
    (str(internal_movie_id), movie_id_to_genres.get(internal_movie_id + 1, ["Unknown"]))  # Adjust back by adding 1
    for internal_movie_id in item_map  # Use internal movie IDs from item_map
]

# Step 4: Build the item features
try:
    item_features = dataset.build_item_features(item_features_list)
    print(f"Successfully built item features with {len(item_features_list)} entries.")
except ValueError as e:
    print(f"Error during item feature building: {e}")

Missing movie IDs: [200950, 200952, 200956, 200960, 200962, 200964, 200968, 200972, 200974, 200978, 200980, 200982, 200984, 200986, 200988, 200990, 200994, 200998, 201000, 201002, 201004, 201006, 201008, 201010, 201012, 201014, 201016, 201018, 201022, 201024, 201026, 201030, 201034, 201036, 201038, 201040, 201042, 201044, 201046, 201048, 201050, 201052, 201054, 201056, 201058, 201062, 201064, 201066, 201068, 201070, 201072, 201074, 201076, 201078, 201080, 201084, 201086, 201088, 201090, 201094, 201096, 201098, 201100, 201104, 201106, 201108, 201110, 201112, 201114, 201118, 201120, 201122, 201124, 201128, 201130, 201132, 201134, 201136, 201138, 201140, 201142, 201144, 201146, 201148, 201150, 201152, 201154, 201156, 201158, 201160, 201162, 201164, 201166, 201168, 201170, 201172, 201174, 201176, 201180, 201182, 201186, 201188, 201192, 201194, 201196, 201198, 201200, 201202, 201204, 201208, 201212, 201216, 201218, 201220, 201222, 201226, 201228, 201230, 201232, 201234, 201236, 201238, 2012

In [27]:
missing_from_fit = ratings_movie_ids - movies_movie_ids
missing_from_genres = ratings_movie_ids - genre_movie_ids

print(f"Movie IDs in ratings but missing in movies: {len(missing_from_fit)}")
print(f"Sample missing movie IDs from movies: {list(missing_from_fit)[:10]}")

print(f"Movie IDs in ratings but missing in genres: {len(missing_from_genres)}")
print(f"Sample missing movie IDs from genres: {list(missing_from_genres)[:10]}")

Movie IDs in ratings but missing in movies: 0
Sample missing movie IDs from movies: []
Movie IDs in ratings but missing in genres: 0
Sample missing movie IDs from genres: []


In [21]:
# Make sure IDs are strings and ratings are numeric
user_ids = ratings['userId'].values
movie_ids = ratings['movieId'].values
ratings_values = ratings['rating'].values

# Build interactions using zip
(interactions, weights) = dataset.build_interactions(
    zip(user_ids, movie_ids, ratings_values)
)

In [22]:
# Print a few mappings for verification
# user_map, item_map, user_id_map = dataset.mapping()
item_id_map = dataset.mapping()[2]
inv_item_id_map = {v: k for k, v in item_id_map.items()}
registered_movie_ids = list(item_id_map.keys())

print("Sample internal → raw item IDs:")
for i in range(10):
    raw_id = inv_item_id_map.get(i)
    print(f"{i} → {raw_id}")


Sample internal → raw item IDs:
0 → 1
1 → 2
2 → 3
3 → 4
4 → 5
5 → 6
6 → 7
7 → 8
8 → 9
9 → 10


In [11]:
sparse.save_npz("interactions_matrix.npz", interactions)

# Save weights matrix
sparse.save_npz("weights_matrix.npz", weights)

In [24]:
# Now safe to build item features
item_features_list = [
    (movie_id, movie_id_to_genres[movie_id])
    for movie_id in item_id_map  # these are the movies LightFM expects
]
item_features = dataset.build_item_features(item_features_list)


# Check shapes
print("Number of items in fit():", len(item_ids_in_fit))
print("Item features shape:", item_features.shape)


KeyError: 116054

In [13]:
user_id_map, item_id_map, user_feature_map, item_feature_map = dataset.mapping()
print("Known movieIds in dataset:", list(item_id_map.keys())[:10])

Known movieIds in dataset: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [14]:
for movie_id in movie_id_to_genres:
    if movie_id not in item_id_map:
        print("Missing movieId in dataset:", movie_id)


Missing movieId in dataset: 200950
Missing movieId in dataset: 200952
Missing movieId in dataset: 200956
Missing movieId in dataset: 200960
Missing movieId in dataset: 200962
Missing movieId in dataset: 200964
Missing movieId in dataset: 200968
Missing movieId in dataset: 200972
Missing movieId in dataset: 200974
Missing movieId in dataset: 200978
Missing movieId in dataset: 200980
Missing movieId in dataset: 200982
Missing movieId in dataset: 200984
Missing movieId in dataset: 200986
Missing movieId in dataset: 200988
Missing movieId in dataset: 200990
Missing movieId in dataset: 200994
Missing movieId in dataset: 200998
Missing movieId in dataset: 201000
Missing movieId in dataset: 201002
Missing movieId in dataset: 201004
Missing movieId in dataset: 201006
Missing movieId in dataset: 201008
Missing movieId in dataset: 201010
Missing movieId in dataset: 201012
Missing movieId in dataset: 201014
Missing movieId in dataset: 201016
Missing movieId in dataset: 201018
Missing movieId in d

In [11]:
model = LightFM(loss = "warp")
model.fit(interactions, item_features = item_features, epochs=3, num_threads=2)

In [13]:
with open("lightfm_model.pkl", "wb") as f:
    pickle.dump(model,f)

with open("lightfm_dataset.pkl", "wb") as f:
    pickle.dump(dataset, f)

In [4]:
with open("lightfm_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("lightfm_dataset.pkl", "rb") as f:
    dataset = pickle.load(f)

interactions = sparse.load_npz("interactions_matrix.npz")
weights = sparse.load_npz("weights_matrix.npz")

In [12]:
def recommend_movies_lightfm(user_id, movies_df, n_recommendations=10):
    """
    Recommend movies for a given user using a trained LightFM model.

    Parameters:
        user_id (str or int): The ID of the user to recommend movies for.
        model (LightFM): A trained LightFM model.
        dataset (Dataset): The fitted LightFM dataset object.
        movies_df (pd.DataFrame): DataFrame with movieId and title columns.
        n_recommendations (int): Number of recommendations to return.

    Returns:
        pd.DataFrame: Top recommended movie titles.
    """
    user_id = int(user_id)  # Ensure it's string like in dataset fitting

    # Get mapping dictionaries
    user_map, item_map, user_id_map, item_id_map = dataset.mapping()

    # Check if user exists in mapping
    if user_id not in user_map:
        print(f"User {user_id} not found in dataset.")
        return pd.DataFrame()

    user_internal_id = user_map[user_id]

    n_items = len(item_map)
    scores = model.predict(user_internal_id, np.arange(n_items), item_features=item_features)
    top_items = np.argsort(-scores)[:n_recommendations]

    # Manually invert item_map to get internal → raw mapping
    inv_item_map = {v: k for k, v in item_map.items()}

    recommended_movies = []
    for internal_id in top_items:
        raw_movie_id = inv_item_map.get(internal_id)
        if raw_movie_id is not None:
            raw_movie_id_int = int(raw_movie_id)  # Convert string to int
            title_row = movies_df[movies_df['movieId'] == raw_movie_id_int]
            if not title_row.empty:
                title = title_row.iloc[0]['title']
                recommended_movies.append((raw_movie_id_int, title))

    return pd.DataFrame(recommended_movies, columns=["movieId", "title"])


In [13]:
test_trial = recommend_movies_lightfm(100,movies_df= movies, n_recommendations=10)
test_trial

Exception: Number of item feature rows does not equal the number of items

In [32]:
print(f"Number of items in the dataset: {len(item_map)}")
print(f"Shape of item features: {item_features.shape}")
# Ensure that the movie IDs in your dataset map correctly to the internal indices in item_features
item_map_check = dataset.mapping()[1]
print(f"Number of items in item_map: {len(item_map_check)}")
print(f"Shape of item features matrix: {item_features.shape}")



Number of items in the dataset: 200949
Shape of item features: (87585, 87604)
Number of items in item_map: 200949
Shape of item features matrix: (87585, 87604)


In [33]:
# For every item in the dataset, ensure the corresponding feature exists
item_features_list = []
for item_id in item_map:
    features = movie_id_to_genres.get(item_id, [])
    item_features_list.append((item_id, features))
item_features = dataset.build_item_features(item_features_list)


ValueError: item id 91 not in item id mappings.

In [35]:
# Check which movie_ids are in the dataset's item mapping
item_map = dataset.mapping()[1]

# Check if all movie_ids in movie_id_to_genres are in the dataset's item_map
missing_movie_ids = [movie_id for movie_id in movie_id_to_genres if str(movie_id) not in item_map]
if missing_movie_ids:
    print("Missing movie IDs:", missing_movie_ids)
else:
    print("All movie IDs from movie_id_to_genres are in the dataset.")


Missing movie IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 21